# 1. RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# 2. [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 2.1 설치
- `pip install ragas rapidfuzz`

## 2.2 RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 2.3 주요 평가지표
### 2.3.1 Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### 2.3.1.1 Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 2.3.1.1.1평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### 2.3.2 Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 2.3.2.1 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## 2.4 Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### 2.4.1 Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 2.4.1.1평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 2.4.1.2 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### 2.4.2 Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 2.4.2.1 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# 3. RAGAS 평가 실습

In [5]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

In [6]:
# docker run -p 6333:6333 -p 6334:6334   -v qdrant_storage:/qdrant/storage   qdrant/qdrant
# termianal에서 실행

## 0. 패키지 설치

In [7]:
# 필요한 library: pymupdf pdfplumber pillow openai qdrant-client langchain-core langchain-openai langchain-qdrant langchain-text-splitters python-dotenv


## 1. 설정 및 클라이언트 준비


In [8]:
import os
import re
import glob
import uuid
from pathlib import Path

import fitz  # PyMuPDF
import pdfplumber
from PIL import Image
from openai import OpenAI
from dotenv import load_dotenv

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# ----- 설정값 -----
GUIDANCE_DIR = "./길라잡이"          # 이 폴더 안의 모든 PDF를 찾아서 적재

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY") or None
COLLECTION_NAME = "guidance_vectordb"

CHAT_MODEL = "gpt-5.4-mini"                          # 표 요약 / 이미지 캐법션용
EMBED_MODEL = "text-embedding-3-large"         # large 임베딩 모델
EMBED_DIM = 3072

CHUNK_SIZE = 800       # 텍스트 청크 최대 글자 수 (splitter 기준)
CHUNK_OVERLAP = 150    # 청크 간 격치는 글자 수 (문맥 끓김 방지)

openai_client = OpenAI()
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)


def ensure_collection() -> bool:
    """collection이 이미 있으면 True(기존 데이터 존재), 없으면 새로 만들고 False를 반환"""
    existing = [c.name for c in qdrant_client.get_collections().collections]
    if COLLECTION_NAME in existing:
        print(f"'{COLLECTION_NAME}' collection이 이미 존재합니다.")
        return True
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )
    print(f"'{COLLECTION_NAME}' collection 생성 완료 (dim={EMBED_DIM})")
    return False


collection_already_existed = ensure_collection()

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

pdf_paths = sorted(glob.glob(str(Path(GUIDANCE_DIR) / "*.pdf")))
print(f"'{GUIDANCE_DIR}' 안에서 발견된 PDF {len(pdf_paths)}개")
for p in pdf_paths:
    print(" -", p)


'guidance_vectordb' collection이 이미 존재합니다.
'./길라잡이' 안에서 발견된 PDF 2개
 - 길라잡이\사병_길라잡이.pdf
 - 길라잡이\초급간부_길라잡이.pdf


## 2. 텍스트 정제

In [9]:
def clean_text(text: str) -> str:
    # Private Use Area (U+E000–U+F8FF) 문자 제거
    text = re.sub(r'[\ue000-\uf8ff]', '', text)
    # 과도한 공백/빈 줄 정리
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


## 3. 표 추출 → 마크다운 → 요약

In [10]:
def table_to_markdown(table_rows):
    rows = [[cell if cell is not None else "" for cell in row] for row in table_rows]
    if not rows:
        return ""
    header = rows[0]
    body = rows[1:]
    md_lines = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    for row in body:
        md_lines.append("| " + " | ".join(row) + " |")
    return "\n".join(md_lines)


def extract_tables(pdf_path: str):
    tables_out = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                if not table or len(table) < 2:
                    continue
                md_table = table_to_markdown(table)
                tables_out.append({"page": page_num, "markdown": md_table})
    return tables_out


def summarize_table(table_markdown: str) -> str:
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        max_completion_tokens=300,
        messages=[{
            "role": "user",
            "content": (
                "다음은 문서에서 추출한 표입니다. 이 표가 어떤 내용을 담고 있는지 "
                "검색에 활용할 수 있도록 2~4문장의 한국어로 요약해줘. "
                "컬럼이 의미하는 바와 주요 수치/항목을 포함해줘.\n\n"
                f"{table_markdown}"
            ),
        }],
    )
    return response.choices[0].message.content.strip()


## 4. 텍스트 청킹 (페이지 단위)

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

# 이 길이를 넘는 페이지만 splitter로 추가 분할 (그 외엔 페이지 통째로 한 청크)
PAGE_OVERFLOW_THRESHOLD = 1200


def chunk_text_by_page(pdf_path: str) -> list[dict]:
    """페이지 단위로 청크를 만든다. 페이지가 너무 길 때만 그 페이지 안에서 추가로 나눈다."""
    doc = fitz.open(pdf_path)
    results = []

    for page_num, page in enumerate(doc, start=1):
        page_text = clean_text(page.get_text("text"))

        if not page_text:
            continue  # 텍스트 없는 페이지(이미지만 있는 페이지 등)는 건너뜀

        if len(page_text) <= PAGE_OVERFLOW_THRESHOLD:
            results.append({"page": page_num, "text": page_text})
        else:
            # 이 페이지만 너무 길어서 임베딩이 흐려질 수 있는 경우 -> 페이지 내부에서만 분할
            sub_chunks = text_splitter.split_text(page_text)
            for sub in sub_chunks:
                results.append({"page": page_num, "text": sub})

    doc.close()
    return results


## 5. PDF 1개 → LangChain `Document` 리스트로 통합

In [12]:
ID_NAMESPACE = uuid.NAMESPACE_URL


def make_id(doc_name: str, chunk_type: str, index: int) -> str:
    """같은 PDF를 다시 적재해도 같은 ID가 나오도록 결정론적으로 생성 (재실행 시 upsert로 덮어씀)"""
    return str(uuid.uuid5(ID_NAMESPACE, f"{doc_name}::{chunk_type}::{index}"))


def build_documents_for_pdf(pdf_path: str) -> tuple[list[Document], list[str]]:
    doc_name = Path(pdf_path).name
    documents = []
    ids = []
    counter = 0

    # 1) 텍스트 청크
    text_chunks = chunk_text_by_page(pdf_path)
    for c in text_chunks:
        documents.append(Document(
            page_content=c["text"],
            metadata={"doc_name": doc_name, "page": c["page"], "type": "text"},
        ))
        ids.append(make_id(doc_name, "text", counter))
        counter += 1

    # 2) 표
    tables = extract_tables(pdf_path)
    for t in tables:
        summary = summarize_table(t["markdown"])
        documents.append(Document(
            page_content=summary,
            metadata={
                "doc_name": doc_name, "page": t["page"], "type": "table",
                "table_markdown": t["markdown"], "summary": summary,
            },
        ))
        ids.append(make_id(doc_name, "table", counter))
        counter += 1

    print(f"[{doc_name}] 텍스트 {len(text_chunks)} / 표 {len(tables)} → 총 {len(documents)}개")
    return documents, ids


## 6. 디렉토리 전체 적재 (upsert)

In [13]:
def upsert_documents(documents: list[Document], ids: list[str], batch_size: int = 100):
    for i in range(0, len(documents), batch_size):
        batch_docs = documents[i:i + batch_size]
        batch_ids = ids[i:i + batch_size]
        vectorstore.add_documents(documents=batch_docs, ids=batch_ids)
        print(f"  upsert 진행: {min(i + batch_size, len(documents))}/{len(documents)}")


def run_pipeline(directory: str = GUIDANCE_DIR):
    paths = sorted(glob.glob(str(Path(directory) / "*.pdf")))
    print(f"'{directory}'에서 PDF {len(paths)}개 발견, 적재 시작\n")

    total_chunks = 0
    for pdf_path in paths:
        documents, ids = build_documents_for_pdf(pdf_path)
        upsert_documents(documents, ids)
        total_chunks += len(documents)
        print()

    print(f"완료: PDF {len(paths)}개 / 총 {total_chunks}개 청크 적재")


# collection이 이미 있었다면 재적재를 건너뜀 (PDF 파싱/표 요약/임베딩 API 재호출 비용 방지)
if not collection_already_existed:
    run_pipeline(GUIDANCE_DIR)
else:
    print(f"'{COLLECTION_NAME}' 컴렉션에 이미 데이터가 있어 적재를 건너뜁니다. (재적재하려면 run_pipeline(GUIDANCE_DIR)를 직접 호출하세요)")


'guidance_vectordb' 컴렉션에 이미 데이터가 있어 적재를 건너뜁니다. (재적재하려면 run_pipeline(GUIDANCE_DIR)를 직접 호출하세요)


## 7. 검색 예시

In [14]:
def search(query: str, k: int = 5, type_filter: str = None):
    qfilter = None
    if type_filter:
        qfilter = Filter(must=[FieldCondition(key="metadata.type", match=MatchValue(value=type_filter))])

    results = vectorstore.similarity_search_with_score(query, k=k, filter=qfilter)

    for doc, score in results:
        print(f"[score={score:.3f}] doc={doc.metadata.get('doc_name')} page={doc.metadata.get('page')} type={doc.metadata.get('type')}")
        print(doc.page_content[:200])
        print("-" * 40)

    return results


# 사용 예시:
search("군인 해택 휴양시설알려줘")
# search("휴가 관련 표", type_filter="table")
# search("기차 할인 안내 이미지", type_filter="image")



[score=0.585] doc=사병_길라잡이.pdf page=23 type=text
2026 병 복지 길라잡이 23
 • 국군복지단 휴양시설 현황
구분
시 설 명
위 치
전화번호(군)
전화번호(일반)
직영
서귀포 호텔
제주 서귀포시
960-7703
064-738-0123
화진포 콘도
강원 고성군 
960-7706
033-682-0500
청간정 콘도
강원 고성군 
822-6470
033-631-9331
송 정 콘도
강원 강릉시 
940-38
----------------------------------------
[score=0.549] doc=사병_길라잡이.pdf page=22 type=text
Part 01 생활편의 분야
22 2026 병 복지 길라잡이
휴양시설 이용
이용대상 / 방법 및 혜택
 • 이용대상 : 현역병, 배우자 및 그 직계 존·비속
 * 가족의 경우 가족관계 증빙서류, 신분증 또는 국방가족 모바일증명 제시
 * 국방가족 모바일증명은 국군복지포털 홈페이지를 이용하여 등록
 • 국군복지단 휴양시설 이용방법
 - 국군복지포털 홈페이지 
----------------------------------------
[score=0.543] doc=초급간부_길라잡이.pdf page=35 type=text
34 
2026 초급간부 길라잡이
•국군복지단 휴양시설 현황
구분
시 설 명
위 치
전화번호(군)
전화번호(일반)
직영
서귀포 호텔
제주 서귀포시
960-7703
064-738-0123
화진포 콘도
강원 고성군 
960-7706
033-682-0500
청간정 콘도
강원 고성군 
988-7873
033-631-9331
송정 콘도
강원 강릉시 
940-3881
----------------------------------------
[score=0.525] doc=초급간부_길라잡이.pdf page=34 type=text
02
Part
생
활
편
의
 
분
야
33 
2026 초급간부 길라잡이
•국군복지단 초급간부전용 객실예약 이용방법
- 국군복지포털 호텔/

[(Document(metadata={'doc_name': '사병_길라잡이.pdf', 'page': 23, 'type': 'text', '_id': 'fb4aa98d-c931-5260-82b6-e1fa8da946b5', '_collection_name': 'guidance_vectordb'}, page_content='2026 병 복지 길라잡이 23\n • 국군복지단 휴양시설 현황\n구분\n시 설 명\n위 치\n전화번호(군)\n전화번호(일반)\n직영\n서귀포 호텔\n제주 서귀포시\n960-7703\n064-738-0123\n화진포 콘도\n강원 고성군 \n960-7706\n033-682-0500\n청간정 콘도\n강원 고성군 \n822-6470\n033-631-9331\n송 정 콘도\n강원 강릉시 \n940-3881\n033-652-7573\n대 천 콘도\n충남 보령시 \n960-7705\n041-932-6305\n민영\n한 화\n설악, 용인, 대천, 산정호수, \n해운대, 경주, 제주, 평창, 거제\n국군복지단\n예약실\n984-6584\n국군복지단\n예약실\n1577-9800\n내선 “1번”\n금 호\n통영, 화순, 설악, 제주, 아산\n소 노\n설악, 경주, 단양, 홍천, 양양, \n변산, 거제, 삼척, 진도, 여수, \n고양(호텔), 양평\n켄싱턴\n남원, 설악비치, 충주, 경주,\n가평, 설악밸리, 하동, 서귀포\n무 주\n전북무주\n리 솜\n안면도\n엘도라도\n전남신안\n샤인빌(소노캄)\n제주\n보 광\n제주\n웰리힐리\n강원횡성\n영랑호\n강원 속초\n현대수\n강원 속초\n • 각 군 휴양시설 이용방법 : 각 군 홈페이지 참조\n구분\n시 설 명\n홈페이지 주소\n국방부\n국방컨벤션(웨딩)\nwww.mndconvention.co.kr\n육군\n로카우스 호텔, 계룡스파텔\nwelfare.army.mil.kr\n해군\n해군호텔(서울, 평택, 제주), \n해군회관(진해)\nwelfare.navy.mil.kr\n공군\n공군호텔\nwww.airforcehote

## 8. Retriever 생성

In [15]:
##################################################
# retriever 생성
##################################################

def get_retriever(vectorstore, k: int = 5):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever


retriever = get_retriever(vectorstore)
print(retriever)


tags=['QdrantVectorStore', 'OpenAIEmbeddings'] vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x000002911D0A8590> search_kwargs={'k': 5}


In [16]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

prompt_txt = """<instruction>
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
"""
prompt = ChatPromptTemplate.from_template(template=prompt_txt)

model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()


def format_docs_for_prompt(documents: list[Document]) -> str:
    """LLM 프롬프트({context})용: 문서들을 하나의 문자열로 합침 (doc_name/page도 같이 표기)"""
    parts = []
    for doc in documents:
        meta = doc.metadata
        header = f"[{meta.get('doc_name', '')} p.{meta.get('page', '')}]"
        parts.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


def format_docs_for_ragas(documents: list[Document]) -> list[str]:
    """RAGAS 평가용: 검색된 문서 각각의 내용을 list[str]로 추출"""
    return [doc.page_content for doc in documents]


def build_prompt_input(state: dict) -> dict:
    """검색 결과({'documents':..., 'query':...})를 prompt 입력({'context':str, 'query':str})으로 변환"""
    return {
        "context": format_docs_for_prompt(state["documents"]),
        "query": state["query"],
    }


# 1단계: 질문으로 검색 → 원본 Document 리스트 + 질문 보존 (retriever는 여기서 1회만 호출됨)
retrieve_step = {
    "documents": retriever,
    "query": RunnablePassthrough(),
}

# RAG 체인 -> RAG 평가 데이터셋을 만드는 RAG Chain -> 최종응답: LLM 응답(str), 검색한 문서들(list[str]), 질문(str)
chain = (
    RunnablePassthrough()  # dict | dict 가 파이썬 dict 병합 연산으로 취급되는 것을 막기 위해 필요
    | retrieve_step
    | {
        "response": RunnableLambda(build_prompt_input) | prompt | model | parser,
        "retrieved_context": RunnableLambda(lambda state: format_docs_for_ragas(state["documents"])),
        "query": RunnableLambda(lambda state: state["query"]),
    }
)


In [17]:
res = chain.invoke("병장의 월급은?")

In [18]:
res["response"]

'병장의 월급은 **1,500,000원**입니다.'

In [19]:
res["retrieved_context"]

['Part 01 생활편의 분야\n6 2026 병 복지 길라잡이\n보 수\n봉 급\n • 계급별 지급액\n 단위 : 원\n \n이병\n일병\n상병\n병장\n750,000\n900,000\n1,200,000\n1,500,000\n • 병 봉급 인상\n 단위 : 원\n연도\n계급\n’21년\n’22년\n’23년\n’24년\n’25~’26년\n병 장 \n608,500\n676,100\n1,000,000\n1,250,000\n1,500,000\n상 병 \n549,200\n610,200\n800,000\n1,000,000\n1,200,000\n일 병\n496,900\n552,100\n680,000\n800,000\n900,000\n이 병\n459,100\n510,100\n600,000\n640,000\n750,000\n인상율(병장)\n12.5%\n11.1%\n47.9%\n25%\n20%\n* 병역의무 이행자의 보상과 예우를 위해 병 봉급과 자산형성프로그램을 결합하여\n’25년까지 병장 기준 200만원 이상으로 인상\n수 당\n단위 : 원\n \n구 분\n지급 기준\n금액\n특수지\n근 무\n수 당\n갑\n・비무장지대, 서해5도, 울릉도, 접적 해역 상주 근무자\n * 가산금 : 서해5도 40,000원, 비무장지대 및 \n북방한계선 인접해역 20,000원\n25,000 \n(가산금별도)\n을\n・GOP, 해안초소, 해발 800m 이상 고지대 상주 근무자\n * 가산금 : 10,000원(GOP / 해안초소)\n20,000\n(가산금별도)\n항 공\n수 당\n을\n・항공기 동승근무자\n40,000 \n함 정\n수 당\n을\n・전투함 및 지원함 승무원\n * 함정출동가산금 : 1일 4,000원\n32,700 \n(가산금별도)\n병\n・수륙양용궤도차량 승무원\n32,700 \npart 01\n생활편의 분야\n▶▶▶ 병 복지혜택 이렇게 다양합니다',
 '이 표는 **계급별 연도(’21~’26년) 급여 수준과 병장 급여 인상률**을 비교한 자료입니다. 행은 **병장·상병·일병·이병

# 4. RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## 4.1 TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.


> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 > 넣는다.
>    ```python
>       try:
>          from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [20]:
# !uv pip install langchain_google_vertexai

In [21]:
# 주피터노트북 환경에서 비동기적 처리 위해 
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [22]:
from ragas.testset import TestsetGenerator

In [23]:
#
# testset -> Context들(문서들) - [질문 - 정답답변 + (Retriever가 찾은 문서 + LLM 응답: chain에서 생성)]
# 1. Context(문서들)을 추출 - TestsetGenerator -> 질문과 정답 답변 생성

import random

# 데이터셋을 생성할 때 사용할 Context를 추출
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "guidance_vectordb"

info = client.get_collection(COLLECTION_NAME)
total_docs = info.points_count

results, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=total_docs,
)

# random하게 k(5)개를 sampling
sample_docs:"list[PointStruct]" =random.sample(results, 10) # 리스트에서 랜덤하게 k(5)개를 추출

# PointStruct - payload: page_content, metadata
# page_content만 추출해서 list[str]
docs = [point.payload['page_content'] for point in sample_docs]



In [24]:
total_docs

337

In [25]:
sample_docs

[Record(id='037f8624-432a-566d-ac6b-f517ddc3e748', payload={'page_content': '이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으로 나눈 내용으로 보입니다. 컬럼은 **‘구분’**(항목/기준), **‘장교’**(장교 대상 적용 비율), **‘부사관’**(부사관 대상 적용 비율)을 의미하며, 표에 제시된 수치로는 장교의 경우 **2년차 60%, 3년차 20%, 5~7년차 20%**, **3년차 70%, 4년차 30%**, **2년차 30%, 3년차 40%, 4년차 30%** 등 연차별 비율이 나타납니다. 또한 일부 항목은 **‘이후 결원 발생 시’**라고 되어 있어, 정해진 연차 비율 적용 후에는 결원 발생 시 추가 충원하는 방식임을 보여줍니다.', 'metadata': {'doc_name': '초급간부_길라잡이.pdf', 'page': 11, 'type': 'table', 'table_markdown': '| 구 분 | 장 교 | 부사관 |\n| --- | --- | --- |\n|  | 2년차 60%, 3년차 20%,\n5년차∼7년차 20% |  |\n|  | 3년차 70%, 4년차 30%,\n이후 결원발생 시 |  |\n|  | 3년차 70%, 4년차 30%,\n이후 결원발생 시 |  |\n|  | 2년차 30%, 3년차 40%,\n4년차 30%, 이후 결원발생 시 |  |', 'summary': '이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으로 나눈 내용으로 보입니다. 컬럼은 **‘구분’**(항목/기준), **‘장교’**(장교 대상 적용 비율), **‘부사관’**(부사관 대상 적용 비율)을 의미하며, 표에 제시된 수치로는 장교의 경우 **2년차 60%, 3년차 20%, 5~7년차 20%**, **3년차 70%, 4년차 30%**, **2년차 30%, 3년차 40%, 4년차 30%** 등 연차별 비율이 나타납니다. 또한 일부 항목은 

In [26]:
docs

['이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으로 나눈 내용으로 보입니다. 컬럼은 **‘구분’**(항목/기준), **‘장교’**(장교 대상 적용 비율), **‘부사관’**(부사관 대상 적용 비율)을 의미하며, 표에 제시된 수치로는 장교의 경우 **2년차 60%, 3년차 20%, 5~7년차 20%**, **3년차 70%, 4년차 30%**, **2년차 30%, 3년차 40%, 4년차 30%** 등 연차별 비율이 나타납니다. 또한 일부 항목은 **‘이후 결원 발생 시’**라고 되어 있어, 정해진 연차 비율 적용 후에는 결원 발생 시 추가 충원하는 방식임을 보여줍니다.',
 '46 \n2026 초급간부 길라잡이\n 개 념 \n\x07하사 이상 현역간부, 군무원 및 공무원 등 개인에게 배정된 복지예산 한도 내에서 \n개인의 욕구와 필요에 따라 복지항목 및 수혜수준을 선택할 수 있는 제도\n* \x07맞춤형복지 예산은 기본항목(단체보험)을 선 계약 후 남은 금액을 개인별 복지점수에 따라 \n자율항목(복지포인트)으로 분기마다 지급\n* \x07맞춤형복지(단체보험, 복지포인트) 관련 자세한 내용은 “IMND 복지포털(웹/APP)”에서 확인 \n가능\n 맞춤형복지 구성 항목\n구분\n기본항목 (단체보험)\n자율항목 (복지포인트)\n내용\n•생명/상해 보험 \n•입·통원 의료비 보장 보험\n•매분기별 복지포인트 지급\n•개인별 자율적 선택사용\n기타\n•’26년 단체보험 계약현황\n - 주 계약 보험사 : 한화손해보험\n - 공동 서비스제공 : \x07삼성화재, 롯데손해보험, \n농협손해보험(자녀)\n•매년 11월 30일까지만 \n 사용 가능 \n (미사용 잔액은 불용처리)\n 이용방법 \n*\x07 \x07주의사항 : 복지포인트 미사용(불용)액 최소화를 위해 신규 임관자 대상 카드 발급 및 11월 말 \n까지 사용토록 관심 필요\n*\x07 \x07단체보험 중 생명·상해 보험은 의무가입 사항이며, 실손보험의 경우 개인실손 가입자에 한해 

In [27]:
# testset 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#testsetgenerator 는 gpt-5 이후 버전은 사용할 수 없다. 
## Langchain의 모델과 Embedding 모델 -> ragas에서 사용할 수 있도록 변환(wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="""
- 사람들이 현역병 복지에 대해서 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법읋 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
""" 
# 질문/답변을 생성할 때 LLM에게 전달할 system Prompt를 설정
)





C:\Users\Playdata\AppData\Local\Temp\ipykernel_15048\3702810314.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15048\3702810314.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [28]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Context 내용 테스트셋 개수(질문 답변 개수)
)

Applying CustomNodeFilter:   0%|          | 0/10 [00:00<?, ?it/s]Node e17f73cf-cd99-461f-8b93-de0d66134e63 does not have a summary. Skipping filtering.
Node c5fedc65-e467-4f05-861d-e2b018c3865c does not have a summary. Skipping filtering.
Node 5d85ee0b-3bba-4d30-94c8-7c37e171fe5c does not have a summary. Skipping filtering.
Node 7644745e-9170-48a6-b275-1ce554c3ae86 does not have a summary. Skipping filtering.
Node a5505d13-5df9-4f8f-bc9b-4a1f48c45d72 does not have a summary. Skipping filtering.
Node f60e778b-9a8c-4a52-bd09-7286d873a818 does not have a summary. Skipping filtering.
Node 90b34060-764c-4072-a064-5d6950042544 does not have a summary. Skipping filtering.
Node 1358ae46-78d3-40e5-b96b-1a95b3a411d1 does not have a summary. Skipping filtering.
Node 30ce3bec-18c3-49f2-8f5e-a646c7caa648 does not have a summary. Skipping filtering.
Node 84cd5f46-5fec-484e-888c-2ee4965c0814 does not have a summary. Skipping filtering.
Generating Samples: 100%|██████████| 12/12 [00:02<00:00,  4.21it/

In [29]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='부사관의 연차별 승진 또는 보직 비율은 어떻게 되나요?', retrieved_contexts=None, reference_contexts=['이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으로 나눈 내용으로 보입니다. 컬럼은 **‘구분’**(항목/기준), **‘장교’**(장교 대상 적용 비율), **‘부사관’**(부사관 대상 적용 비율)을 의미하며, 표에 제시된 수치로는 장교의 경우 **2년차 60%, 3년차 20%, 5~7년차 20%**, **3년차 70%, 4년차 30%**, **2년차 30%, 3년차 40%, 4년차 30%** 등 연차별 비율이 나타납니다. 또한 일부 항목은 **‘이후 결원 발생 시’**라고 되어 있어, 정해진 연차 비율 적용 후에는 결원 발생 시 추가 충원하는 방식임을 보여줍니다.'], retrieved_context_ids=None, reference_context_ids=None, response=None, multi_responses=None, reference='표에 따르면 부사관의 연차별 승진 또는 보직 비율은 구분별로 제시되어 있으며, 일부 항목에서는 정해진 연차 비율 적용 후 결원 발생 시 추가 충원하는 방식이 적용됩니다.', rubrics=None, persona_name='Military Personnel Planner', query_style='WEB_SEARCH_LIKE', query_length='MEDIUM'), synthesizer_name='single_hop_specific_query_synthesizer'), TestsetSample(eval_sample=SingleTurnSample(user_input='농협손해보험은 맞춤형복지제도에서 어떤 역할을 하나요?', retrieved_contexts=None, reference_contexts=['46 

In [30]:
sample1 = testset.samples[0].eval_sample
print("사용자질문:", sample1.user_input)
print("Context:", sample1.reference_contexts)
print("생성된 답변(정답):", sample1.reference)
print("평가대상 RAG의 답변:", sample1.response)
print("평가대상 RAG가 검색한 Context:", sample1.retrieved_contexts)


사용자질문: 부사관의 연차별 승진 또는 보직 비율은 어떻게 되나요?
Context: ['이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으로 나눈 내용으로 보입니다. 컬럼은 **‘구분’**(항목/기준), **‘장교’**(장교 대상 적용 비율), **‘부사관’**(부사관 대상 적용 비율)을 의미하며, 표에 제시된 수치로는 장교의 경우 **2년차 60%, 3년차 20%, 5~7년차 20%**, **3년차 70%, 4년차 30%**, **2년차 30%, 3년차 40%, 4년차 30%** 등 연차별 비율이 나타납니다. 또한 일부 항목은 **‘이후 결원 발생 시’**라고 되어 있어, 정해진 연차 비율 적용 후에는 결원 발생 시 추가 충원하는 방식임을 보여줍니다.']
생성된 답변(정답): 표에 따르면 부사관의 연차별 승진 또는 보직 비율은 구분별로 제시되어 있으며, 일부 항목에서는 정해진 연차 비율 적용 후 결원 발생 시 추가 충원하는 방식이 적용됩니다.
평가대상 RAG의 답변: None
평가대상 RAG가 검색한 Context: None


In [31]:
# 생성된 Testset을 Pandas DataFrame으로 변환
eval_df = testset.to_pandas()

In [32]:
eval_df.shape

(12, 7)

In [33]:
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,부사관의 연차별 승진 또는 보직 비율은 어떻게 되나요?,[이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으...,"표에 따르면 부사관의 연차별 승진 또는 보직 비율은 구분별로 제시되어 있으며, 일부...",Military Personnel Planner,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
1,농협손해보험은 맞춤형복지제도에서 어떤 역할을 하나요?,"[46 \n2026 초급간부 길라잡이\n 개 념 \n하사 이상 현역간부, 군무원 ...",농협손해보험은 맞춤형복지제도의 단체보험 항목에서 자녀를 대상으로 공동 서비스제공 보...,HR Benefits Coordinator,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
2,복지정책협력 담당 부서는 어디인가요?,[02\nPart\n생\n활\n편\n의\n \n분\n야\n47 \n2026 초급간부...,복지정책협력 담당부서는 국방부 복지정책과입니다.,HR Benefits Coordinator,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
3,평택에 위치한 국방 관련 복지·숙박 시설에는 무엇이 있습니까?,"[이 표는 국방 관련 복지·숙박 시설 목록을 정리한 것으로 보이며, 컬럼은 각각 *...","해군호텔(서울·평택·제주) 및 해군회관(진해)이 포함되어 있으며, 이 중 평택에 위...",Military Personnel Planner,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
4,사회적응 지원 분야에서 청년장병 뉴스타트 프로그램은 어떤 대상에게 어떤 지원을 제공...,[<1-hop>\n\n취업 및 창업 지원\n장병 경제교육\n국민연금 군 복무 크레딧...,사회적응 지원 분야에서 청년장병 뉴스타트 프로그램은 군 복무 중인 전 장병과 청년장...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,장병 경제교욱이랑 사이버아카데미 뭐가 다르나요?,[<1-hop>\n\n취업 및 창업 지원\n장병 경제교육\n국민연금 군 복무 크레딧...,장병 경제교육은 군 복무 중인 장병을 대상으로 경제 관련 교육을 제공하는 프로그램이...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,사회적응 지원 뭐고 청년장병 뉴스타트 프로그램이랑 무슨 관계 있어요?,[<1-hop>\n\n취업 및 창업 지원\n장병 경제교육\n국민연금 군 복무 크레딧...,사회적응 지원은 군 복무 중이나 전역 예정 장병들이 취업이나 창업 준비할 수 있게 ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,"사회적응 지원 분야에서 청년장병 뉴스타트 프로그램이 뭐고, 이 프로그램이 군 복무 ...",[<1-hop>\n\n취업 및 창업 지원\n장병 경제교육\n국민연금 군 복무 크레딧...,사회적응 지원 분야에는 청년장병 뉴스타트 프로그램이 포함되어 있어. 이 프로그램은 ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,"해군 복무연장 지원하려면 어떤 절차 따라야 하고, 해군에서 지원 가능한 특기병 종류...",[<1-hop>\n\n01\nPart\n인\n사\n제\n도\n \n근\n무\n \n...,"해군 복무연장 지원하려면 먼저 선발계획 공지 보고, 지원서 접수하고, 부대추천 심사...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,"해군 복무연장 지원하려면 언제 신청해야 하고, 해군에서 조리병이나 갑판병 같은 특기...",[<1-hop>\n\n01\nPart\n인\n사\n제\n도\n \n근\n무\n \n...,"해군 복무연장 지원은 연 2회(전반기, 후반기) 진행되며, 임관 2년차부터 9년차(...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [34]:
row_idx = 0
q = eval_df.loc[row_idx, 'user_input']
resp = chain.invoke(q)


In [35]:
# resp
resp['response']

'부사관의 연차별 승진 또는 보직 비율은 다음과 같습니다.\n\n- **2년차 30%**\n- **3년차 40%**\n- **4년차 30%**\n\n또한 일부 항목은 **이후 결원 발생 시**로 제시되어 있어, 정해진 연차 비율 적용 후에는 결원 발생 시 추가 충원하는 방식입니다.'

In [36]:
# resp
resp['retrieved_context']

['이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으로 나눈 내용으로 보입니다. 컬럼은 **‘구분’**(항목/기준), **‘장교’**(장교 대상 적용 비율), **‘부사관’**(부사관 대상 적용 비율)을 의미하며, 표에 제시된 수치로는 장교의 경우 **2년차 60%, 3년차 20%, 5~7년차 20%**, **3년차 70%, 4년차 30%**, **2년차 30%, 3년차 40%, 4년차 30%** 등 연차별 비율이 나타납니다. 또한 일부 항목은 **‘이후 결원 발생 시’**라고 되어 있어, 정해진 연차 비율 적용 후에는 결원 발생 시 추가 충원하는 방식임을 보여줍니다.',
 '이 표는 **각 군(육군·해군·공군)의 장교/부사관 장기복무 선발 제도**를 정리한 것으로, 장기복무 선발 시 얻는 혜택과 함께 **평가요소 및 배점 기준**을 보여줍니다. 혜택으로는 **소령(상사) 진출 기회**, **계급별 정년(소령 50세, 상사 53세)**, **군사교육 선발 조건**, **국외연수·국방무관·UN PKO 등 해외근무 기회**가 포함됩니다.  \n\n표의 핵심 컬럼은 **구분(군/계급/복무연차)**, **계(총점)**, 그리고 **상훈·근무평정·교육성적·부대평가·추천·면접·체력검정·잠재역량평가·기술/자격·위원회 심사** 같은 세부 평가항목이며, 총점은 대체로 **100점(공군 부사관은 90점)** 기준으로 구성됩니다. 예를 들어 육군 장교는 **2~3년차 100점**, **5년차 이상 100점**으로 나뉘고, 해군 장교/부사관과 공군 장교/부사',
 '이 표는 각 계급 간 진급을 위한 **최소 복무기간**과 **연령 상한**을 정리한 기준표입니다. 컬럼은 **병장→하사, 하사→중사, 중사→상사, 중위→대위, 대위→소령, 소령→중령**의 진급 구간을 뜻하며, 예를 들어 병장→하사는 **병장으로 2년(예비역 1년차부터)**, 하사→중사는 **7년**, 중사→상사는 **12년** 복무가 필요합니다. 연령 기준은 각 구간별로 **병장→하사 만

In [37]:
# 모든 testset의 데이터데 대한 llm 응답과 retriever의 검색 겷과를 추가
response_list = []
retrieved_context_list = []

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp["response"])
    retrieved_context_list.append(resp["retrieved_context"])

In [38]:
print(len(response_list), len(retrieved_context_list))

12 12


In [39]:
response_list

['정보가 부족해 답을 할 수없습니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '평택에 관련된 국방 복지·숙박 시설로는 **해군호텔(평택)**이 있습니다.  \n해당 내용은 Context의 표에서 **해군호텔(서울, 평택, 제주)**로 제시되어 있습니다.',
 '청년장병 뉴스타트 프로그램은 **당해 연도 전역예정장병인 청년장병(간부 중심)**을 대상으로 하며, **취업역량 강화와 취업률 향상**을 위해 **채용 단계별 프로그램**을 제공합니다.\n\n주요 지원은 다음과 같습니다.\n- **초기상담 및 컨설팅 안내**\n- **진로방향·목표점검**\n- **역량·강점 분석**\n- **입사서류 클리닉**\n- **모의면접**\n- **채용정보 제공**\n- **직무·기업분석**\n- **희망기업 알선**\n- **직장적응지원**\n- **경력개발지원**\n- **취업성공사례관리**\n- **취업기업관리**\n\n또한 유형별로는:\n- **뉴 스타트Ⅰ형**: 취업역량 기본과정\n- **뉴 스타트Ⅱ형**: 취업준비 심화과정, 1:1 컨설팅 및 취업매칭 지원\n- **뉴 스타트Ⅲ형**: 채용 설명회, 일자리 매칭데이·공직 채용 설명회·창업 설명회 제공\n\n',
 '장병 경제교육과 사이버아카데미는 **목적과 교육 내용이 다릅니다.**\n\n- **장병 경제교육**:  \n  병사 봉급 인상에 따른 **자산 형성, 재무관리, 신용관리, 금융사기 예방** 등을 위한 경제교육입니다.  \n  대상은 **육·해·공군, 해병대 장병 및 훈련기관 교육생**이고, **오프라인/온라인**으로 운영됩니다.\n\n- **사이버아카데미**:  \n  전역예정장병의 **전직준비와 생애설계 역량 강화**를 위한 **온라인 상시학습시스템**입니다.  \n  취업역량, 진로설정, 전직준비, 직무, 자격, 어학, AI디지털역량 등 **취업·창업 준비 중심**의 교육을 제공합니다.\n\n즉,  \n**장병 경제교육은 ‘금융·경제 관리’ 중심**,

In [40]:
retrieved_context_list

[['이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으로 나눈 내용으로 보입니다. 컬럼은 **‘구분’**(항목/기준), **‘장교’**(장교 대상 적용 비율), **‘부사관’**(부사관 대상 적용 비율)을 의미하며, 표에 제시된 수치로는 장교의 경우 **2년차 60%, 3년차 20%, 5~7년차 20%**, **3년차 70%, 4년차 30%**, **2년차 30%, 3년차 40%, 4년차 30%** 등 연차별 비율이 나타납니다. 또한 일부 항목은 **‘이후 결원 발생 시’**라고 되어 있어, 정해진 연차 비율 적용 후에는 결원 발생 시 추가 충원하는 방식임을 보여줍니다.',
  '이 표는 **각 군(육군·해군·공군)의 장교/부사관 장기복무 선발 제도**를 정리한 것으로, 장기복무 선발 시 얻는 혜택과 함께 **평가요소 및 배점 기준**을 보여줍니다. 혜택으로는 **소령(상사) 진출 기회**, **계급별 정년(소령 50세, 상사 53세)**, **군사교육 선발 조건**, **국외연수·국방무관·UN PKO 등 해외근무 기회**가 포함됩니다.  \n\n표의 핵심 컬럼은 **구분(군/계급/복무연차)**, **계(총점)**, 그리고 **상훈·근무평정·교육성적·부대평가·추천·면접·체력검정·잠재역량평가·기술/자격·위원회 심사** 같은 세부 평가항목이며, 총점은 대체로 **100점(공군 부사관은 90점)** 기준으로 구성됩니다. 예를 들어 육군 장교는 **2~3년차 100점**, **5년차 이상 100점**으로 나뉘고, 해군 장교/부사관과 공군 장교/부사',
  '이 표는 각 계급 간 진급을 위한 **최소 복무기간**과 **연령 상한**을 정리한 기준표입니다. 컬럼은 **병장→하사, 하사→중사, 중사→상사, 중위→대위, 대위→소령, 소령→중령**의 진급 구간을 뜻하며, 예를 들어 병장→하사는 **병장으로 2년(예비역 1년차부터)**, 하사→중사는 **7년**, 중사→상사는 **12년** 복무가 필요합니다. 연령 기준은 각 구간별로 **병장→하

In [41]:
#
# eval_df에 컬럼으로 추가
#
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,response,retrieved_contexts
0,부사관의 연차별 승진 또는 보직 비율은 어떻게 되나요?,[이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으...,"표에 따르면 부사관의 연차별 승진 또는 보직 비율은 구분별로 제시되어 있으며, 일부...",Military Personnel Planner,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으...
1,농협손해보험은 맞춤형복지제도에서 어떤 역할을 하나요?,"[46 \n2026 초급간부 길라잡이\n 개 념 \n하사 이상 현역간부, 군무원 ...",농협손해보험은 맞춤형복지제도의 단체보험 항목에서 자녀를 대상으로 공동 서비스제공 보...,HR Benefits Coordinator,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[46 \n2026 초급간부 길라잡이\n 개 념 \n하사 이상 현역간부, 군무원 ..."
2,복지정책협력 담당 부서는 어디인가요?,[02\nPart\n생\n활\n편\n의\n \n분\n야\n47 \n2026 초급간부...,복지정책협력 담당부서는 국방부 복지정책과입니다.,HR Benefits Coordinator,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[02\nPart\n생\n활\n편\n의\n \n분\n야\n47 \n2026 초급간부...
3,평택에 위치한 국방 관련 복지·숙박 시설에는 무엇이 있습니까?,"[이 표는 국방 관련 복지·숙박 시설 목록을 정리한 것으로 보이며, 컬럼은 각각 *...","해군호텔(서울·평택·제주) 및 해군회관(진해)이 포함되어 있으며, 이 중 평택에 위...",Military Personnel Planner,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,평택에 관련된 국방 복지·숙박 시설로는 **해군호텔(평택)**이 있습니다. \n해...,"[이 표는 국방 관련 복지·숙박 시설 목록을 정리한 것으로 보이며, 컬럼은 각각 *..."
4,사회적응 지원 분야에서 청년장병 뉴스타트 프로그램은 어떤 대상에게 어떤 지원을 제공...,[<1-hop>\n\n취업 및 창업 지원\n장병 경제교육\n국민연금 군 복무 크레딧...,사회적응 지원 분야에서 청년장병 뉴스타트 프로그램은 군 복무 중인 전 장병과 청년장...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer,청년장병 뉴스타트 프로그램은 **당해 연도 전역예정장병인 청년장병(간부 중심)**을...,[62 \n2026 초급간부 길라잡이\n-③-Ⅲ\n청년장병 뉴 스타트(New sta...
5,장병 경제교욱이랑 사이버아카데미 뭐가 다르나요?,[<1-hop>\n\n취업 및 창업 지원\n장병 경제교육\n국민연금 군 복무 크레딧...,장병 경제교육은 군 복무 중인 장병을 대상으로 경제 관련 교육을 제공하는 프로그램이...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer,장병 경제교육과 사이버아카데미는 **목적과 교육 내용이 다릅니다.**\n\n- **...,[2026 병 복지 길라잡이 69\n장병 경제교육\n개 요\n • 병사 봉급의 급격...
6,사회적응 지원 뭐고 청년장병 뉴스타트 프로그램이랑 무슨 관계 있어요?,[<1-hop>\n\n취업 및 창업 지원\n장병 경제교육\n국민연금 군 복무 크레딧...,사회적응 지원은 군 복무 중이나 전역 예정 장병들이 취업이나 창업 준비할 수 있게 ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer,사회적응 지원은 **전역 전후 장병이 사회로 돌아가 적응하도록 돕는 지원 분야**를...,[62 \n2026 초급간부 길라잡이\n-③-Ⅲ\n청년장병 뉴 스타트(New sta...
7,"사회적응 지원 분야에서 청년장병 뉴스타트 프로그램이 뭐고, 이 프로그램이 군 복무 ...",[<1-hop>\n\n취업 및 창업 지원\n장병 경제교육\n국민연금 군 복무 크레딧...,사회적응 지원 분야에는 청년장병 뉴스타트 프로그램이 포함되어 있어. 이 프로그램은 ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer,"청년장병 뉴스타트(New start) 프로그램은 **당해 연도 전역예정 장병, 특히...",[이 표는 **군 복무 중인 장병과 전역 예정 장병을 대상으로 한 취·창업 지원 프...
8,"해군 복무연장 지원하려면 어떤 절차 따라야 하고, 해군에서 지원 가능한 특기병 종류...",[<1-hop>\n\n01\nPart\n인\n사\n제\n도\n \n근\n무\n \n...,"해군 복무연장 지원하려면 먼저 선발계획 공지 보고, 지원서 접수하고, 부대추천 심사...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n13 \n...
9,"해군 복무연장 지원하려면 언제 신청해야 하고, 해군에서 조리병이나 갑판병 같은 특기...",[<1-hop>\n\n01\nPart\n인\n사\n제\n도\n \n근\n무\n \n...,"해군 복무연장 지원은 연 2회(전반기, 후반기) 진행되며, 임관 2년차부터 9년차(...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer,"해군 복무연장은 **연 2회(전반기, 후반기)** 신청합니다. \n다만 **정확한...",[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n13 \n...


In [42]:
# eval_df를 ragas의 평가데이터셋 타입으로 변환
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=12)

In [43]:
#
# 평가
#
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    Faithfulness,
    AnswerRelevancy,
)
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_15048\3378832119.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15048\3378832119.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15048\3378832119.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\User

In [44]:
# 평가할 때 사용할 LLM embedding 모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)

]
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_15048\384477035.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_15048\384477035.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
Evaluating:  56%|█████▋    | 27/48 [03:00<02:15,  6.45s/it]Exception raised in Job[6]: TimeoutError()
Exception raised in Job[16]: TimeoutE

In [48]:
eval_result

{'context_recall': 0.7407, 'llm_context_precision_with_reference': 0.6019, 'faithfulness': 0.2857, 'answer_relevancy': 0.3237}

In [49]:
result_df = eval_result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,부사관의 연차별 승진 또는 보직 비율은 어떻게 되나요?,[이 표는 **구분별로 장교와 부사관의 승진·보직 또는 충원 비율**을 연차 기준으...,정보가 부족해 답을 할 수없습니다.,"표에 따르면 부사관의 연차별 승진 또는 보직 비율은 구분별로 제시되어 있으며, 일부...",1.000000,0.887500,NaN,0.000000
1,농협손해보험은 맞춤형복지제도에서 어떤 역할을 하나요?,"[46 \n2026 초급간부 길라잡이\n 개 념 \n하사 이상 현역간부, 군무원 ...",정보가 부족해 답을 할 수없습니다.,농협손해보험은 맞춤형복지제도의 단체보험 항목에서 자녀를 대상으로 공동 서비스제공 보...,1.000000,1.000000,NaN,0.000000
2,복지정책협력 담당 부서는 어디인가요?,[02\nPart\n생\n활\n편\n의\n \n분\n야\n47 \n2026 초급간부...,정보가 부족해 답을 할 수없습니다.,복지정책협력 담당부서는 국방부 복지정책과입니다.,1.000000,1.000000,0.000000,0.000000
3,평택에 위치한 국방 관련 복지·숙박 시설에는 무엇이 있습니까?,"[이 표는 국방 관련 복지·숙박 시설 목록을 정리한 것으로 보이며, 컬럼은 각각 *...",평택에 관련된 국방 복지·숙박 시설로는 **해군호텔(평택)**이 있습니다. \n해...,"해군호텔(서울·평택·제주) 및 해군회관(진해)이 포함되어 있으며, 이 중 평택에 위...",1.000000,0.866667,NaN,0.872953
4,사회적응 지원 분야에서 청년장병 뉴스타트 프로그램은 어떤 대상에게 어떤 지원을 제공...,[62 \n2026 초급간부 길라잡이\n-③-Ⅲ\n청년장병 뉴 스타트(New sta...,청년장병 뉴스타트 프로그램은 **당해 연도 전역예정장병인 청년장병(간부 중심)**을...,사회적응 지원 분야에서 청년장병 뉴스타트 프로그램은 군 복무 중인 전 장병과 청년장...,NaN,0.916667,NaN,0.758829
5,장병 경제교욱이랑 사이버아카데미 뭐가 다르나요?,[2026 병 복지 길라잡이 69\n장병 경제교육\n개 요\n • 병사 봉급의 급격...,장병 경제교육과 사이버아카데미는 **목적과 교육 내용이 다릅니다.**\n\n- **...,장병 경제교육은 군 복무 중인 장병을 대상으로 경제 관련 교육을 제공하는 프로그램이...,NaN,NaN,NaN,0.790361
6,사회적응 지원 뭐고 청년장병 뉴스타트 프로그램이랑 무슨 관계 있어요?,[62 \n2026 초급간부 길라잡이\n-③-Ⅲ\n청년장병 뉴 스타트(New sta...,사회적응 지원은 **전역 전후 장병이 사회로 돌아가 적응하도록 돕는 지원 분야**를...,사회적응 지원은 군 복무 중이나 전역 예정 장병들이 취업이나 창업 준비할 수 있게 ...,1.000000,1.000000,NaN,0.618574
7,"사회적응 지원 분야에서 청년장병 뉴스타트 프로그램이 뭐고, 이 프로그램이 군 복무 ...",[이 표는 **군 복무 중인 장병과 전역 예정 장병을 대상으로 한 취·창업 지원 프...,"청년장병 뉴스타트(New start) 프로그램은 **당해 연도 전역예정 장병, 특히...",사회적응 지원 분야에는 청년장병 뉴스타트 프로그램이 포함되어 있어. 이 프로그램은 ...,NaN,0.950000,1.000000,0.843476
8,"해군 복무연장 지원하려면 어떤 절차 따라야 하고, 해군에서 지원 가능한 특기병 종류...",[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n13 \n...,정보가 부족해 답을 할 수없습니다.,"해군 복무연장 지원하려면 먼저 선발계획 공지 보고, 지원서 접수하고, 부대추천 심사...",0.500000,0.000000,0.000000,0.000000
9,"해군 복무연장 지원하려면 언제 신청해야 하고, 해군에서 조리병이나 갑판병 같은 특기...",[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n13 \n...,"해군 복무연장은 **연 2회(전반기, 후반기)** 신청합니다. \n다만 **정확한...","해군 복무연장 지원은 연 2회(전반기, 후반기) 진행되며, 임관 2년차부터 9년차(...",0.500000,0.000000,0.714286,0.000000
